# Project V — Computational Discovery
## Milestone 4: Cross-Domain Validation of the Candidate-Enriched GMM Component

This notebook tests whether the 32-star M2/M3 GMM reference component has supporting coherence in information not already used to construct the GMM. The validation is deliberately conservative: the GMM was built from `[Fe/H]`, radial velocity, tangential velocity, colour, and absolute magnitude, so those quantities are descriptive context rather than independent evidence.

The main held-out domains checked here are Project II orbital diagnostics, Project III population labels, Project IV chemical-readiness fields, and Project VI validation-risk metadata. Current held-out domain tables cover the 27 known candidates, not the full 1,838-star parent sample; that coverage limit is treated as a scientific result rather than hidden.


In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception as exc:
    SCIPY_AVAILABLE = False
    stats = None
    warnings.warn(f"scipy unavailable; inferential tests will be omitted: {exc}")

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'processed'
FIG = ROOT / 'figures'
REPORT = ROOT / 'report'
FIG.mkdir(exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(exist_ok=True)

OUT_MEMBERSHIP = DATA / 'project_v_gmm_cross_domain_membership.csv'
OUT_COVERAGE = DATA / 'project_v_gmm_cross_domain_coverage_summary.csv'
OUT_GROUP = DATA / 'project_v_gmm_cross_domain_group_summary.csv'
OUT_STATS = DATA / 'project_v_gmm_cross_domain_statistics.csv'
OUT_EVIDENCE = DATA / 'project_v_gmm_cross_domain_evidence_assessment.csv'

FIG_ORBIT = FIG / 'project_v_gmm_cross_domain_orbital_comparison.png'
FIG_POP = FIG / 'project_v_gmm_cross_domain_population_composition.png'
FIG_STAB = FIG / 'project_v_gmm_cross_domain_stability_evidence.png'

RNG_SEED = 42
BOOT_N = 2000


## Load Inputs and Verify the Locked M2/M3 Reference Definition

The reference component is the M2 GMM component with `gmm_label == 5`. It contains 32 stars: 24 known candidates and 8 additional members. The three known candidates outside this component are the M2 omitted candidates.


In [2]:
def source_key(s):
    return pd.Series(s).astype('Int64').astype(str)

def read_csv(name):
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    if 'source_id' in df.columns:
        df['source_id_key'] = source_key(df['source_id'])
    return df

m2 = read_csv('project_v_m2_cluster_assignments.csv')
m3_star = read_csv('project_v_m3_star_stability.csv')
m3_runs = read_csv('project_v_m3_run_summary.csv')
known_candidates = read_csv('project_v_candidate_cross_method_summary.csv')

m2['reference_member'] = m2['gmm_label'].eq(5)
m2['known_candidate'] = m2['known_candidate'].astype(bool)
m2['reference_candidate'] = m2['reference_member'] & m2['known_candidate']
m2['additional_member'] = m2['reference_member'] & ~m2['known_candidate']
m2['omitted_candidate'] = ~m2['reference_member'] & m2['known_candidate']

def assign_group(row):
    if row['reference_candidate']:
        return 'recovered_known_candidate'
    if row['additional_member']:
        return 'additional_gmm_member'
    if row['omitted_candidate']:
        return 'omitted_known_candidate'
    return 'parent_comparison'

m2['m4_group'] = m2.apply(assign_group, axis=1)

baseline = m3_runs.loc[m3_runs['experiment_family'].eq('baseline_reproduction')].iloc[0]
summary_counts = {
    'parent_sample_n': int(len(m2)),
    'known_candidates_n': int(m2['known_candidate'].sum()),
    'reference_component_n': int(m2['reference_member'].sum()),
    'reference_candidates_n': int(m2['reference_candidate'].sum()),
    'additional_members_n': int(m2['additional_member'].sum()),
    'omitted_candidates_n': int(m2['omitted_candidate'].sum()),
    'baseline_ari': float(baseline['ari_vs_m2_gmm']),
    'baseline_candidate_enrichment': float(baseline['candidate_enrichment']),
}
summary_counts


In [3]:
assert summary_counts['parent_sample_n'] == 1838
assert summary_counts['known_candidates_n'] == 27
assert summary_counts['reference_component_n'] == 32
assert summary_counts['reference_candidates_n'] == 24
assert summary_counts['additional_members_n'] == 8
assert summary_counts['omitted_candidates_n'] == 3
assert math.isclose(summary_counts['baseline_ari'], 1.0, rel_tol=0, abs_tol=1e-12)
assert math.isclose(summary_counts['baseline_candidate_enrichment'], 51.05555555555556, rel_tol=1e-9)
print(json.dumps(summary_counts, indent=2))


{
  "parent_sample_n": 1838,
  "known_candidates_n": 27,
  "reference_component_n": 32,
  "reference_candidates_n": 24,
  "additional_members_n": 8,
  "omitted_candidates_n": 3,
  "baseline_ari": 1.0,
  "baseline_candidate_enrichment": 51.05555555555555
}


## Join Audit

All joins are performed on `source_id`. The held-out Project II/III/IV/VI tables are candidate-level products with 27 unique keys, so they cannot independently validate the 8 additional GMM members or the non-candidate parent comparison sample.


In [4]:
validation_sources = {
    'project_ii_orbit_am_consistency': 'project_ii_orbit_angular_momentum_consistency.csv',
    'project_iii_population': 'project_iii_population_candidates.csv',
    'project_iv_metallicity': 'project_iv_metallicity_candidates.csv',
    'project_vi_uncertainty': 'project_vi_uncertainty_inventory.csv',
}

def audit_table(name, df):
    keys = df['source_id_key'] if 'source_id_key' in df.columns else pd.Series(dtype=str)
    return {
        'source_name': name,
        'rows': len(df),
        'unique_source_ids': int(keys.nunique()) if len(keys) else 0,
        'duplicate_source_ids': int(len(keys) - keys.nunique()) if len(keys) else 0,
        'missing_source_ids': int(keys.isna().sum()) if len(keys) else 0,
        'scope_note': 'candidate-level validation table' if len(df) == 27 else 'parent-level table',
    }

source_frames = {name: read_csv(fname) for name, fname in validation_sources.items()}
join_audit = pd.DataFrame([audit_table('m2_parent_assignments', m2), audit_table('m3_star_stability', m3_star), audit_table('project_v_known_candidate_summary', known_candidates)] + [audit_table(k, v) for k, v in source_frames.items()])
join_audit


In [5]:
# Keep one authoritative copy of each held-out column and prefix only where names would collide.
orbit_cols = [
    'source_id_key', 'orbit_am_consistency_label', 'am_expected_orbit_behavior',
    'orbital_family_interpretation', 'rotation_class', 'inclination_proxy_class',
    'Lz_kpc_kms', 'Lperp_kpc_kms', 'Ltot_kpc_kms', 'galpy_eccentricity',
    'galpy_rperi_kpc', 'galpy_rap_kpc', 'galpy_zmax_kpc', 'galpy_energy',
    'galpy_orbit_radial_flag', 'galpy_orbit_high_zmax_flag',
    'galpy_orbit_inner_halo_like_flag', 'galpy_orbit_disk_confined_flag',
    'distance_recovery_source', 'distance_quality_flag', 'galpy_success', 'galpy_potential',
]
pop_cols = [
    'source_id_key', 'project_iii_population_label', 'project_iii_population_group',
    'project_iii_population_confidence', 'project_iii_population_notes',
]
chem_cols = [
    'source_id_key', 'project_iv_metallicity_class', 'project_iv_chemical_readiness',
    'project_iv_chemical_followup_priority', 'project_iv_chemical_notes',
]
vi_cols = [
    'source_id_key', 'project_vi_validation_priority', 'project_vi_validation_risk',
    'project_vi_uncertainty_notes',
]
stability_cols = [
    'source_id_key', 'selection_count', 'selection_frequency',
    'mean_membership_probability', 'median_membership_probability', 'stability_group'
]

base_cols = ['source_id', 'source_id_key', 'feh', 'rv', 'tangential_velocity_kms', 'bp_rp', 'absolute_g_mag',
             'known_candidate', 'gmm_label', 'gmm_membership_probability', 'reference_member',
             'reference_candidate', 'additional_member', 'omitted_candidate', 'm4_group']

membership = m2[base_cols].merge(m3_star[stability_cols], on='source_id_key', how='left', validate='one_to_one')
membership = membership.merge(source_frames['project_ii_orbit_am_consistency'][orbit_cols], on='source_id_key', how='left', validate='one_to_one')
membership = membership.merge(source_frames['project_iii_population'][pop_cols], on='source_id_key', how='left', validate='one_to_one')
membership = membership.merge(source_frames['project_iv_metallicity'][chem_cols], on='source_id_key', how='left', validate='one_to_one')
membership = membership.merge(source_frames['project_vi_uncertainty'][vi_cols], on='source_id_key', how='left', validate='one_to_one')

membership['has_orbital_validation'] = membership['orbit_am_consistency_label'].notna()
membership['has_population_validation'] = membership['project_iii_population_group'].notna()
membership['has_chemical_readiness'] = membership['project_iv_chemical_readiness'].notna()
membership['has_validation_risk'] = membership['project_vi_validation_risk'].notna()

membership['cross_domain_coverage_count'] = membership[['has_orbital_validation','has_population_validation','has_chemical_readiness','has_validation_risk']].sum(axis=1)

# Preserve integer source ids as text-safe keys for downstream catalogue use.
membership.sort_values(['reference_member','known_candidate','selection_frequency'], ascending=[False, False, False], inplace=True)
membership.to_csv(OUT_MEMBERSHIP, index=False)
print(OUT_MEMBERSHIP, membership.shape)
membership.groupby('m4_group')[['has_orbital_validation','has_population_validation','has_chemical_readiness','has_validation_risk']].sum()


/Users/liors/Documents/research/gaia-lamost-galactic-archaeology/data/processed/project_v_gmm_cross_domain_membership.csv (1838, 57)


## Coverage Summary

Coverage is reported separately for the 24 recovered known candidates, 8 additional GMM members, 3 omitted known candidates, and the remaining parent comparison stars.


In [6]:
group_order = ['recovered_known_candidate', 'additional_gmm_member', 'omitted_known_candidate', 'parent_comparison']
domain_cols = {
    'orbital_dynamics': 'has_orbital_validation',
    'population_labels': 'has_population_validation',
    'chemical_readiness': 'has_chemical_readiness',
    'validation_uncertainty': 'has_validation_risk',
}
source_for_domain = {
    'orbital_dynamics': 'project_ii_orbit_angular_momentum_consistency.csv',
    'population_labels': 'project_iii_population_candidates.csv',
    'chemical_readiness': 'project_iv_metallicity_candidates.csv',
    'validation_uncertainty': 'project_vi_uncertainty_inventory.csv',
}
notes_for_domain = {
    'orbital_dynamics': 'Held-out relative to the GMM feature space, but available only for the 27 known candidates.',
    'population_labels': 'Derived from Project II orbital diagnostics and available only for known candidates.',
    'chemical_readiness': 'Fe/H-based readiness is descriptive because Fe/H was a GMM input; no detailed abundances are present.',
    'validation_uncertainty': 'Validation-risk metadata are candidate-level and not independent physical discovery evidence.',
}
rows = []
for domain, col in domain_cols.items():
    for group in group_order:
        sub = membership[membership['m4_group'].eq(group)]
        rows.append({
            'validation_domain': domain,
            'source_file': source_for_domain[domain],
            'join_key': 'source_id',
            'group': group,
            'group_n': int(len(sub)),
            'available_n': int(sub[col].sum()),
            'missing_n': int(len(sub) - sub[col].sum()),
            'coverage_fraction': float(sub[col].mean()) if len(sub) else np.nan,
            'coverage_note': notes_for_domain[domain],
        })
coverage = pd.DataFrame(rows)
coverage.to_csv(OUT_COVERAGE, index=False)
coverage


## Group Summaries

Group summaries report held-out orbital/population quantities where available. Statistics involving the 8 additional GMM members are explicitly missing in candidate-only validation domains.


In [7]:
def q25(x): return np.nanpercentile(x, 25) if pd.Series(x).notna().any() else np.nan
def q75(x): return np.nanpercentile(x, 75) if pd.Series(x).notna().any() else np.nan

def summarize_continuous(df, group, var):
    x = pd.to_numeric(df[var], errors='coerce').dropna()
    return {
        'group': group,
        'metric': var,
        'n': int(len(x)),
        'median': float(np.nanmedian(x)) if len(x) else np.nan,
        'iqr_low': float(q25(x)) if len(x) else np.nan,
        'iqr_high': float(q75(x)) if len(x) else np.nan,
        'minimum': float(np.nanmin(x)) if len(x) else np.nan,
        'maximum': float(np.nanmax(x)) if len(x) else np.nan,
    }

continuous_vars = ['selection_frequency', 'Lz_kpc_kms', 'Lperp_kpc_kms', 'Ltot_kpc_kms',
                   'galpy_eccentricity', 'galpy_zmax_kpc', 'galpy_rperi_kpc', 'galpy_rap_kpc', 'galpy_energy']
summary_rows = []
for group in group_order:
    sub = membership[membership['m4_group'].eq(group)]
    for var in continuous_vars:
        summary_rows.append(summarize_continuous(sub, group, var))

group_summary = pd.DataFrame(summary_rows)

# Add categorical compositions in a long format.
cat_vars = ['rotation_class', 'inclination_proxy_class', 'orbit_am_consistency_label',
            'orbital_family_interpretation', 'project_iii_population_group',
            'project_iii_population_label', 'project_iv_metallicity_class',
            'project_vi_validation_risk', 'distance_quality_flag']
cat_rows = []
for group in group_order:
    sub = membership[membership['m4_group'].eq(group)]
    for var in cat_vars:
        counts = sub[var].fillna('not_available').value_counts(dropna=False)
        for level, n in counts.items():
            cat_rows.append({
                'group': group,
                'metric': var,
                'category': level,
                'n': int(n),
                'fraction': float(n / len(sub)) if len(sub) else np.nan,
            })
cat_summary = pd.DataFrame(cat_rows)

# Write continuous and categorical summaries into one CSV with compatible columns.
group_summary['category'] = np.nan
group_summary['fraction'] = np.nan
group_summary = pd.concat([group_summary, cat_summary], ignore_index=True, sort=False)
group_summary.to_csv(OUT_GROUP, index=False)
group_summary.head(20)


## Statistical Comparisons

Only the 24 recovered known candidates and 3 omitted known candidates have candidate-level orbital/population/chemical-readiness coverage. Tests below are therefore exploratory and are not interpreted as discovery-level evidence.


In [8]:
def cliffs_delta(x, y):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    y = np.asarray(pd.Series(y).dropna(), dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = sum(float(a > b) for a in x for b in y)
    lt = sum(float(a < b) for a in x for b in y)
    return (gt - lt) / (len(x) * len(y))

def bootstrap_median_ci(x, n_boot=BOOT_N, seed=RNG_SEED):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    meds = [np.median(rng.choice(x, size=len(x), replace=True)) for _ in range(n_boot)]
    return tuple(np.percentile(meds, [2.5, 97.5]))

stat_rows = []
recovered = membership[membership['m4_group'].eq('recovered_known_candidate')]
omitted = membership[membership['m4_group'].eq('omitted_known_candidate')]
additional = membership[membership['m4_group'].eq('additional_gmm_member')]

for var in ['selection_frequency', 'Lz_kpc_kms', 'Lperp_kpc_kms', 'Ltot_kpc_kms', 'galpy_eccentricity', 'galpy_zmax_kpc', 'galpy_rperi_kpc', 'galpy_rap_kpc', 'galpy_energy']:
    x = pd.to_numeric(recovered[var], errors='coerce').dropna()
    y = pd.to_numeric(omitted[var], errors='coerce').dropna()
    x_ci = bootstrap_median_ci(x)
    y_ci = bootstrap_median_ci(y)
    p = np.nan
    test = 'not_run'
    if SCIPY_AVAILABLE and len(x) >= 2 and len(y) >= 2:
        p = float(stats.mannwhitneyu(x, y, alternative='two-sided').pvalue)
        test = 'mann_whitney_u_two_sided'
    stat_rows.append({
        'comparison': 'recovered_known_candidate_vs_omitted_known_candidate',
        'metric': var,
        'test': test,
        'n_a': int(len(x)),
        'n_b': int(len(y)),
        'median_a': float(np.median(x)) if len(x) else np.nan,
        'median_b': float(np.median(y)) if len(y) else np.nan,
        'median_a_ci95_low': x_ci[0],
        'median_a_ci95_high': x_ci[1],
        'median_b_ci95_low': y_ci[0],
        'median_b_ci95_high': y_ci[1],
        'effect_size_cliffs_delta': cliffs_delta(x, y),
        'p_value': p,
        'interpretation_note': 'Exploratory only; omitted group has n=3.'
    })

# Fisher exact tests for selected binary held-out flags.
for flag in ['galpy_orbit_radial_flag', 'galpy_orbit_high_zmax_flag', 'galpy_orbit_inner_halo_like_flag', 'galpy_orbit_disk_confined_flag']:
    a_true = int(recovered[flag].fillna(False).astype(bool).sum())
    a_false = int(recovered[flag].notna().sum() - a_true)
    b_true = int(omitted[flag].fillna(False).astype(bool).sum())
    b_false = int(omitted[flag].notna().sum() - b_true)
    p = np.nan
    test = 'not_run'
    if SCIPY_AVAILABLE and (a_true+a_false) and (b_true+b_false):
        p = float(stats.fisher_exact([[a_true, a_false], [b_true, b_false]], alternative='two-sided').pvalue)
        test = 'fisher_exact_two_sided'
    stat_rows.append({
        'comparison': 'recovered_known_candidate_vs_omitted_known_candidate',
        'metric': flag,
        'test': test,
        'n_a': int(a_true+a_false),
        'n_b': int(b_true+b_false),
        'fraction_a': a_true/(a_true+a_false) if (a_true+a_false) else np.nan,
        'fraction_b': b_true/(b_true+b_false) if (b_true+b_false) else np.nan,
        'p_value': p,
        'interpretation_note': 'Exploratory only; omitted group has n=3.'
    })

# Stability versus evidence relationships within the 27 candidate-covered stars.
covered = membership[membership['has_orbital_validation']].copy()
for var in ['galpy_eccentricity', 'galpy_zmax_kpc', 'Lz_kpc_kms', 'Lperp_kpc_kms']:
    x = pd.to_numeric(covered['selection_frequency'], errors='coerce')
    y = pd.to_numeric(covered[var], errors='coerce')
    valid = x.notna() & y.notna()
    rho = p = np.nan
    test = 'not_run'
    if SCIPY_AVAILABLE and valid.sum() >= 4:
        rho, p = stats.spearmanr(x[valid], y[valid])
        rho, p = float(rho), float(p)
        test = 'spearman_rank'
    stat_rows.append({
        'comparison': 'selection_frequency_vs_candidate_heldout_metric',
        'metric': var,
        'test': test,
        'n_a': int(valid.sum()),
        'spearman_rho': rho,
        'p_value': p,
        'interpretation_note': 'Selection frequency is derived from GMM perturbation tests; it is not physical evidence by itself.'
    })

statistics = pd.DataFrame(stat_rows)
statistics.to_csv(OUT_STATS, index=False)
statistics


<string>:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:54: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
<string>:54: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=

## Evidence Assessment

The assessment separates support for the recovered known-candidate core from support for the full 32-star GMM component. The 8 additional members currently lack held-out orbital/population coverage.


In [9]:
covered_27 = membership[membership['known_candidate']]
rec_orb_n = int(recovered['has_orbital_validation'].sum())
add_orb_n = int(additional['has_orbital_validation'].sum())
om_orb_n = int(omitted['has_orbital_validation'].sum())
rec_consistent = int(recovered['orbit_am_consistency_label'].eq('consistent').sum())
om_consistent = int(omitted['orbit_am_consistency_label'].eq('consistent').sum())
rec_pop_counts = recovered['project_iii_population_group'].value_counts().to_dict()
om_pop_counts = omitted['project_iii_population_group'].value_counts().to_dict()

assessment_rows = [
    {
        'evidence_domain': 'orbital_dynamics',
        'coverage_statement': f'{rec_orb_n}/24 recovered candidates and {om_orb_n}/3 omitted candidates have Project II orbital coverage; {add_orb_n}/8 additional members have coverage.',
        'support_for_recovered_candidate_core': f'{rec_consistent}/24 recovered candidates are labelled orbit-AM consistent; orbital diagnostics are available for the recovered known-candidate core.',
        'support_for_additional_members': 'No independent Project II orbital coverage for the 8 additional GMM members.',
        'circularity_risk': 'Low for orbital quantities relative to GMM inputs, although some orbital summaries incorporate velocities related to the broader kinematic context.',
        'classification': 'partially_supported_for_candidate_core; inconclusive_for_full_32_member_component',
    },
    {
        'evidence_domain': 'population_labels',
        'coverage_statement': f'Project III labels cover 24/24 recovered candidates, 3/3 omitted candidates, and 0/8 additional members. Recovered groups: {rec_pop_counts}; omitted groups: {om_pop_counts}.',
        'support_for_recovered_candidate_core': 'Recovered known candidates are mostly assigned to halo-like Project III categories, but the omitted candidates also have candidate-level labels and n=3 is too small for strong separation.',
        'support_for_additional_members': 'No Project III population labels for the 8 additional GMM members.',
        'circularity_risk': 'Moderate because Project III labels are derived from Project II orbital diagnostics and include [Fe/H] context in notes.',
        'classification': 'partially_supported_for_candidate_core; inconclusive_for_full_32_member_component',
    },
    {
        'evidence_domain': 'chemical_information',
        'coverage_statement': 'Project IV currently provides Fe/H-based metallicity class and chemical follow-up readiness for the 27 known candidates only.',
        'support_for_recovered_candidate_core': 'The chemical table is useful for follow-up triage but does not provide independent abundance validation because Fe/H was used by the GMM.',
        'support_for_additional_members': 'No Project IV chemical-readiness rows for the 8 additional GMM members.',
        'circularity_risk': 'High for metallicity as validation because Fe/H is one of the five GMM features; detailed abundance ratios are not present.',
        'classification': 'descriptive_not_independent_validation',
    },
    {
        'evidence_domain': 'stability_vs_physics',
        'coverage_statement': 'M3 selection frequencies are available for all 1,838 stars, but held-out physical diagnostics are available only for the 27 known candidates.',
        'support_for_recovered_candidate_core': f'Mean selection frequency: recovered={recovered.selection_frequency.mean():.4f}, additional={additional.selection_frequency.mean():.4f}, omitted={omitted.selection_frequency.mean():.4f}.',
        'support_for_additional_members': 'Additional members have lower mean M3 selection frequency and no current independent held-out orbital/population validation.',
        'circularity_risk': 'Selection frequency is a computational stability diagnostic rather than an external physical measurement.',
        'classification': 'model_dependent_followup_signal_with_limited_cross_domain_support',
    },
    {
        'evidence_domain': 'overall_m4_conclusion',
        'coverage_statement': 'Cross-domain evidence is strong enough to describe the recovered candidate core, but too incomplete to validate the full 32-star GMM component.',
        'support_for_recovered_candidate_core': 'The 24 recovered known candidates retain supporting orbital/population context from prior candidate-level products.',
        'support_for_additional_members': 'The 8 additional GMM members remain follow-up targets requiring new orbital, chemical, uncertainty, and catalogue checks.',
        'circularity_risk': 'The validation deliberately separates held-out orbital/population evidence from GMM-input feature descriptions.',
        'classification': 'partially_supported_for_known_candidate_core; inconclusive_for_additional_members_and_full_component',
    },
]
evidence = pd.DataFrame(assessment_rows)
evidence.to_csv(OUT_EVIDENCE, index=False)
evidence


## Figures


In [10]:
plot_df = membership[membership['has_orbital_validation']].copy()
colors = {
    'recovered_known_candidate': '#1f77b4',
    'omitted_known_candidate': '#d62728',
    'additional_gmm_member': '#2ca02c',
    'parent_comparison': '#bdbdbd',
}
labels = {
    'recovered_known_candidate': 'Recovered known candidates',
    'omitted_known_candidate': 'Omitted known candidates',
    'additional_gmm_member': 'Additional GMM members',
    'parent_comparison': 'Parent comparison',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
for group in ['recovered_known_candidate','omitted_known_candidate']:
    sub = plot_df[plot_df['m4_group'].eq(group)]
    axes[0].scatter(sub['Lz_kpc_kms'], sub['Lperp_kpc_kms'], s=70, alpha=0.85, label=f"{labels[group]} (n={len(sub)})", color=colors[group], edgecolor='white', linewidth=0.7)
axes[0].axvline(0, color='0.35', lw=1, ls='--')
axes[0].set_xlabel(r'$L_z$ [kpc km s$^{-1}$]')
axes[0].set_ylabel(r'$L_{\perp}$ [kpc km s$^{-1}$]')
axes[0].set_title('Angular-momentum plane')
axes[0].legend(frameon=True, fontsize=9)

box_data = [plot_df.loc[plot_df['m4_group'].eq(g), 'galpy_zmax_kpc'].dropna() for g in ['recovered_known_candidate','omitted_known_candidate']]
axes[1].boxplot(box_data, labels=['Recovered\nknown (n=24)', 'Omitted\nknown (n=3)'], patch_artist=True,
                boxprops=dict(facecolor='#e8eef7'), medianprops=dict(color='#111111', linewidth=1.5))
axes[1].set_ylabel(r'$Z_{max}$ [kpc]')
axes[1].set_title('Orbit vertical extent')
fig.suptitle('Project V — Computational Discovery\nHeld-out orbital diagnostics for candidate-covered stars', fontsize=14)
fig.text(0.5, 0.01, 'The 8 additional GMM members have no current candidate-level Project II orbital coverage.', ha='center', fontsize=9)
fig.savefig(FIG_ORBIT, dpi=180, bbox_inches='tight')
plt.show()

pop = membership[membership['m4_group'].isin(['recovered_known_candidate','omitted_known_candidate','additional_gmm_member'])].copy()
pop['population_display'] = pop['project_iii_population_group'].fillna('not_available')
pop_counts = pop.groupby(['m4_group','population_display']).size().unstack(fill_value=0).reindex(['recovered_known_candidate','additional_gmm_member','omitted_known_candidate'])
fig, ax = plt.subplots(figsize=(11, 5.2), constrained_layout=True)
bottom = np.zeros(len(pop_counts))
palette = plt.cm.Set2(np.linspace(0, 1, max(3, pop_counts.shape[1])))
for color, col in zip(palette, pop_counts.columns):
    vals = pop_counts[col].values
    ax.bar(range(len(pop_counts)), vals, bottom=bottom, label=col, color=color, edgecolor='white', linewidth=0.7)
    bottom += vals
ax.set_xticks(range(len(pop_counts)))
ax.set_xticklabels(['Recovered known\n(n=24)', 'Additional GMM\n(n=8)', 'Omitted known\n(n=3)'])
ax.set_ylabel('Number of stars')
ax.set_title('Project V — Computational Discovery\nProject III population-label coverage and composition')
ax.legend(title='Population group', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
for i, total in enumerate(pop_counts.sum(axis=1)):
    ax.text(i, total + 0.25, f'n={int(total)}', ha='center', va='bottom')
fig.savefig(FIG_POP, dpi=180, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), constrained_layout=True)
means = membership[membership['m4_group'].isin(['recovered_known_candidate','additional_gmm_member','omitted_known_candidate'])].groupby('m4_group')['selection_frequency'].agg(['mean','std','count']).reindex(['recovered_known_candidate','additional_gmm_member','omitted_known_candidate'])
axes[0].bar(range(len(means)), means['mean'], yerr=means['std'].fillna(0), color=[colors[g] for g in means.index], edgecolor='white', linewidth=0.7, capsize=4)
axes[0].set_xticks(range(len(means)))
axes[0].set_xticklabels(['Recovered\nknown', 'Additional\nGMM', 'Omitted\nknown'])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Mean M3 selection frequency')
axes[0].set_title('Computational stability by group')
for i, (idx, row) in enumerate(means.iterrows()):
    axes[0].text(i, row['mean'] + 0.04, f"n={int(row['count'])}\n{row['mean']:.3f}", ha='center', va='bottom', fontsize=9)

for group in ['recovered_known_candidate','omitted_known_candidate']:
    sub = plot_df[plot_df['m4_group'].eq(group)]
    axes[1].scatter(sub['selection_frequency'], sub['galpy_eccentricity'], s=70, alpha=0.85, label=f"{labels[group]} (n={len(sub)})", color=colors[group], edgecolor='white', linewidth=0.7)
axes[1].set_xlabel('M3 selection frequency')
axes[1].set_ylabel('Galpy eccentricity')
axes[1].set_title('Stability versus held-out orbit diagnostic')
axes[1].legend(frameon=True, fontsize=9)
fig.suptitle('Project V — Computational Discovery\nComputational stability and cross-domain evidence are not equivalent', fontsize=14)
fig.savefig(FIG_STAB, dpi=180, bbox_inches='tight')
plt.show()


<string>:26: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
<string>:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
<string>:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
<string>:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Compact Results for Reporting


In [11]:
print('Written outputs:')
for path in [OUT_MEMBERSHIP, OUT_COVERAGE, OUT_GROUP, OUT_STATS, OUT_EVIDENCE, FIG_ORBIT, FIG_POP, FIG_STAB]:
    print(path.relative_to(ROOT), path.stat().st_size)

key_results = {
    'coverage_orbit_recovered': f"{int(recovered.has_orbital_validation.sum())}/24",
    'coverage_orbit_additional': f"{int(additional.has_orbital_validation.sum())}/8",
    'coverage_orbit_omitted': f"{int(omitted.has_orbital_validation.sum())}/3",
    'selection_frequency_recovered_mean': float(recovered.selection_frequency.mean()),
    'selection_frequency_additional_mean': float(additional.selection_frequency.mean()),
    'selection_frequency_omitted_mean': float(omitted.selection_frequency.mean()),
    'recovered_population_groups': recovered.project_iii_population_group.value_counts().to_dict(),
    'omitted_population_groups': omitted.project_iii_population_group.value_counts().to_dict(),
    'overall_classification': 'partially_supported_for_known_candidate_core; inconclusive_for_additional_members_and_full_component',
}
print(json.dumps(key_results, indent=2))


Written outputs:
data/processed/project_v_gmm_cross_domain_membership.csv 563659
data/processed/project_v_gmm_cross_domain_coverage_summary.csv 3279
data/processed/project_v_gmm_cross_domain_group_summary.csv 8916
data/processed/project_v_gmm_cross_domain_statistics.csv 4535
data/processed/project_v_gmm_cross_domain_evidence_assessment.csv 3212
figures/project_v_gmm_cross_domain_orbital_comparison.png 148577
figures/project_v_gmm_cross_domain_population_composition.png 100124
figures/project_v_gmm_cross_domain_stability_evidence.png 154881
{
  "coverage_orbit_recovered": "24/24",
  "coverage_orbit_additional": "0/8",
  "coverage_orbit_omitted": "3/3",
  "selection_frequency_recovered_mean": 0.8785569105691057,
  "selection_frequency_additional_mean": 0.41310975609756095,
  "selection_frequency_omitted_mean": 0.056910569105691,
  "recovered_population_groups": {
    "retrograde_halo": 12,
    "radial_halo_or_gse_like": 4,
    "prograde_hot_or_heated_disk": 3,
    "retrograde_uncertain":